# 🌳 Forest Awareness — Proyecto de Machine Learning con CRISP-ML(Q)

**Sistema de alerta temprana de deforestación en municipios de Colombia**

Universidad de Cundinamarca · Ingeniería de Sistemas · Integrantes: Iris Lorena Peñaloza, Julian Santiago Canchon y Scarlet Natalia Niño Ramírez

| | |
|---|---|
| **Repositorio** | https://github.com/scarletnatalianio23-source/Proyect_machine_learning- |
| **Aplicación publicada** | https://forest-awareness-4qpy.onrender.com/ |
| **Datos** | `data/forest_loss_municipalities.csv` (**dataset simulado**, 188 municipios, 2015-2023) |

Este cuaderno documenta **todo el proceso** de las seis fases de CRISP-ML(Q) y es el origen de las cifras, tablas y figuras del informe técnico y de la aplicación Flask.

### Contenido
0. Preparación del entorno
1. **Fase 1** — Comprensión del negocio y de los datos
2. **Fase 2** — Ingeniería de datos (incluye **reducción de dimensionalidad: PCA y t-SNE**)
3. **Fase 3** — Ingeniería del modelo
4. **Fase 4** — Evaluación del modelo
5. **Fase 5** — Despliegue
6. **Fase 6** — Monitoreo y mantenimiento
7. Resumen de consistencia y exportación de resultados

> **Cómo usarlo en Google Colab:** menú *Entorno de ejecución → Ejecutar todo*. El cuaderno clona el repositorio automáticamente, genera las figuras en `static/figures/`, los artefactos en `models/` y las tablas en `reports/tables/`. Al final descarga un `.zip` con todo.

## 0. Preparación del entorno
Se importan las librerías, se clona el repositorio (si no se está dentro de él) y se fija la semilla aleatoria para que **todos los resultados sean reproducibles**.

In [ ]:
import os, sys, json, math, time, warnings, subprocess, zipfile, csv, logging
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

import sklearn
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.manifold import TSNE, trustworthiness
from sklearn.metrics import make_scorer, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
import joblib

REPO_URL = "https://github.com/scarletnatalianio23-source/Proyect_machine_learning-.git"
if not os.path.exists("data/forest_loss_municipalities.csv"):
    if not os.path.exists("Proyect_machine_learning-"):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("Proyect_machine_learning-")
print("Directorio de trabajo:", os.getcwd())
print("scikit-learn:", sklearn.__version__, "| pandas:", pd.__version__, "| numpy:", np.__version__)

for carpeta in ["static/figures", "models", "reports/tables", "logs"]:
    os.makedirs(carpeta, exist_ok=True)

SEED = 2026
np.random.seed(SEED)

VERDE, VERDE_OSC, DORADO, GRIS, ROJO = "#0B6B3A", "#023A22", "#C8A22A", "#6B7A72", "#B5412F"
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

def savefig(nombre):
    plt.tight_layout()
    plt.savefig(f"static/figures/{nombre}.png", dpi=150, bbox_inches="tight")
    plt.show()

def mostrar(texto):
    display(Markdown(texto))

RESULTS = {}   # aquí se acumulan las cifras que usan el informe y la aplicación

---
# FASE 1 · Comprensión del negocio y de los datos

## 1.1 Problema aprobado, contexto y partes interesadas

**Problema.** Colombia tiene cerca del 52 % de su territorio cubierto por bosque natural, pero sigue perdiéndolo: según el IDEAM (SMByC) se deforestaron **119.483 ha en 2025**, un 5,2 % más que en 2024. La pérdida se concentra en departamentos como Meta, Caquetá, Guaviare y Putumayo y se asocia con ganadería extensiva, agroindustria, cultivos ilícitos, minería ilegal e infraestructura. Las cifras oficiales llegan en boletines **después** de que la pérdida ocurrió: las autoridades y comunidades no tienen una herramienta simple que indique *qué municipios perderán más bosque el próximo año, qué tan urgente es y qué podría cambiarlo*.

**Partes interesadas y beneficiarios**

| Parte interesada | Interés | Cómo usa el sistema |
|---|---|---|
| Autoridades ambientales (CAR, alcaldías, Ministerio) | Decidir dónde ubicar equipos de monitoreo escasos | Rol *Authority*: pronóstico, simulador *what-if*, tablero y perfiles de riesgo |
| Comunidades y ciudadanía | Entender el riesgo de su territorio | Rol *Citizen*: pronóstico e historial |
| Administradores de la plataforma | Mantener usuarios, modelo y calidad | Rol *Admin*: gestión de usuarios y reentrenamiento |
| Equipo académico (nosotros) | Aplicar CRISP-ML(Q) de forma trazable | Documentación y código reproducible |

## 1.2 Objetivos de negocio → objetivos de Machine Learning

| Objetivo de negocio | Objetivo de Machine Learning | Métrica |
|---|---|---|
| Anticipar la pérdida de bosque por municipio | Predecir las **hectáreas perdidas el año siguiente** a partir de la historia y las condiciones del municipio | MAE, RMSE (ha) y R² |
| Priorizar municipios con poca capacidad de monitoreo | Clasificar la predicción en **niveles de alerta** (Bajo / Moderado / Alto / Crítico) según cuantiles históricos | Cobertura y sesgo por nivel |
| Comunicar la incertidumbre | Entregar un **rango** de la estimación además de un número | Cobertura empírica del intervalo |
| Explorar escenarios | Mostrar cómo cambia el pronóstico si cambian condiciones productivas y sociales | Sensibilidad del modelo |

## 1.3 Preguntas que responde el modelo y tarea de aprendizaje

1. ¿Cuántas hectáreas de bosque perderá un municipio el próximo año?
2. ¿Qué nivel de alerta corresponde a esa cifra y con qué incertidumbre?
3. ¿Qué factores (historia reciente, ganado, coca, pobreza, grupos armados…) pesan más en el pronóstico?

**Tarea de aprendizaje:** *regresión supervisada* sobre series de tiempo en panel (municipio × año). Variable objetivo: `forest_loss_ha` del año siguiente. Los niveles de alerta son una regla de umbrales posterior a la predicción; el agrupamiento por perfiles de riesgo (K-Means) y la asignación de equipos (aprendizaje por refuerzo) son módulos de apoyo de la aplicación y no forman parte de la evaluación del modelo predictivo.

## 1.4 Fuentes de datos, variables y tamaño

| Variable | Unidad | Papel | Significado |
|---|---|---|---|
| `municipality`, `department`, `region` | texto | identificador | Nombres reales; la región es Andina, Caribe, Pacífico, Orinoquía o Amazonía |
| `year` | año | índice | 2015-2023 |
| `area_km2` | km² | predictor | Superficie aproximada del municipio |
| `population` | personas | predictor | Población (simulada) |
| `cattle_heads` | cabezas | predictor | Ganado (simulado) |
| `planted_area_ha` | ha | predictor | Área sembrada (simulada) |
| `nbi_pct` | % | predictor | Necesidades básicas insatisfechas (simulado) |
| `coca_ha` | ha | predictor | Cultivos de coca (simulado) |
| `armed_groups` | conteo | predictor | Grupos armados presentes (simulado) |
| **`forest_loss_ha`** | ha | **objetivo** | Bosque perdido en el año |

> ⚠️ **El dataset es simulado.** Los nombres de municipios/departamentos y las áreas son reales (aproximadas), pero población, ganado, área sembrada, NBI, coca, grupos armados y pérdida de bosque se generan con `data/generate_dataset.py` (semilla 2026). **No son cifras oficiales** del IDEAM, DANE o SMByC. Está inspirado en: Mendoza Montoya, E. (2025). *Municipal-level data on deforestation, prosecutions, and socioeconomic indicators in Colombia (2021-2023)*. Mendeley Data. https://doi.org/10.17632/mn5s748pyt.1

In [ ]:
df = pd.read_csv("data/forest_loss_municipalities.csv")
print("Dimensiones (filas, columnas):", df.shape)
display(df.head())
display(pd.DataFrame({"tipo": df.dtypes.astype(str), "nulos": df.isna().sum(), "valores únicos": df.nunique()}))
display(df.describe().T.round(1))

## 1.5 Calidad, relevancia y suficiencia de los datos
Se verifican nulos, duplicados, rangos válidos, estructura del panel y consistencia interna.

In [ ]:
claves = ["municipality", "department"]
num_cols = ["area_km2", "population", "cattle_heads", "planted_area_ha", "nbi_pct", "coca_ha", "armed_groups", "forest_loss_ha"]
n_mun = df.groupby(claves).ngroups
filas_por_mun = df.groupby(claves).size().unique().tolist()
area_const = int(df.groupby(claves)["area_km2"].nunique().max())

calidad = pd.DataFrame([
    ["Filas (municipio-año)", len(df)],
    ["Municipios (municipio + departamento)", n_mun],
    ["Departamentos / regiones", f"{df.department.nunique()} / {df.region.nunique()}"],
    ["Rango de años", f"{df.year.min()}-{df.year.max()} ({df.year.nunique()} años)"],
    ["Panel balanceado (filas por municipio)", str(filas_por_mun)],
    ["Valores nulos (total)", int(df.isna().sum().sum())],
    ["Filas duplicadas", int(df.duplicated().sum())],
    ["Valores negativos en variables numéricas", int((df[num_cols] < 0).sum().sum())],
    ["nbi_pct fuera de [0, 100]", int(((df.nbi_pct < 0) | (df.nbi_pct > 100)).sum())],
    ["Área constante dentro de cada municipio (valores distintos máx.)", area_const],
    ["Años con pérdida = 0 ha (%)", round((df.forest_loss_ha == 0).mean() * 100, 1)],
], columns=["Verificación", "Resultado"])
display(calidad)
calidad.to_csv("reports/tables/f1_calidad_datos.csv", index=False)

RESULTS["dataset"] = {"filas": int(len(df)), "municipios": int(n_mun), "anios": [int(df.year.min()), int(df.year.max())],
                      "nulos": int(df.isna().sum().sum()), "duplicados": int(df.duplicated().sum())}

mostrar(f'''**Lectura.** El panel está **completo y balanceado** ({n_mun} municipios × {df.year.nunique()} años = {len(df):,} filas), sin nulos ni duplicados y con rangos válidos.
Esto es esperable porque el dataset es **simulado**: la ausencia de nulos/duplicados es una propiedad del generador, no evidencia de que los datos reales estén limpios.
Por eso la limpieza se documenta como *verificación* (no se inventan errores) y el tratamiento se concentra en lo que sí aparece: **asimetría fuerte y valores extremos** (secciones 1.6 y 2.2).''')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].hist(df.forest_loss_ha, bins=60, color=VERDE)
ax[0].set_title("Pérdida de bosque (ha) — escala original"); ax[0].set_xlabel("ha por municipio-año")
ax[1].hist(np.log1p(df.forest_loss_ha), bins=50, color=DORADO)
ax[1].set_title("log(1 + pérdida) — escala transformada"); ax[1].set_xlabel("log(1 + ha)")
por_region = df.groupby("region")["forest_loss_ha"].mean().sort_values()
ax[2].barh(por_region.index, por_region.values, color=VERDE_OSC)
ax[2].set_title("Pérdida media por región (ha/año)"); ax[2].set_xlabel("ha")
savefig("fig01_distribucion_objetivo")

asim_orig, asim_log = stats.skew(df.forest_loss_ha), stats.skew(np.log1p(df.forest_loss_ha))
RESULTS["asimetria_objetivo"] = {"original": round(float(asim_orig), 2), "log1p": round(float(asim_log), 2)}
mostrar(f"**Asimetría (skewness) del objetivo:** {asim_orig:.2f} en escala original → {asim_log:.2f} tras `log(1+x)`. "
        "La distribución original está muy sesgada a la derecha (muchos municipios con poca pérdida y unos pocos de la Amazonía con miles de hectáreas); "
        "la transformación logarítmica la acerca a una forma más simétrica, adecuada para modelos lineales y para medir error relativo.")

### Valores atípicos (outliers)
Se aplica la regla del rango intercuartílico (IQR): un valor es atípico si está fuera de `[Q1 − 1,5·IQR, Q3 + 1,5·IQR]`.

In [ ]:
def tabla_outliers(frame, columnas):
    filas = []
    for c in columnas:
        q1, q3 = frame[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n = int(((frame[c] < lo) | (frame[c] > hi)).sum())
        filas.append([c, round(lo, 2), round(hi, 2), n, round(n / len(frame) * 100, 1)])
    return pd.DataFrame(filas, columns=["variable", "límite inferior", "límite superior", "n atípicos", "% atípicos"])

cols_out = ["forest_loss_ha", "area_km2", "population", "cattle_heads", "planted_area_ha", "coca_ha", "nbi_pct", "armed_groups"]
out_orig = tabla_outliers(df, cols_out)
df_log = df.copy()
for c in ["forest_loss_ha", "area_km2", "population", "cattle_heads", "planted_area_ha", "coca_ha"]:
    df_log[c] = np.log1p(df_log[c])
out_log = tabla_outliers(df_log, cols_out)

comp = out_orig[["variable", "% atípicos"]].merge(out_log[["variable", "% atípicos"]], on="variable", suffixes=(" (original)", " (log1p)"))
display(comp)
comp.to_csv("reports/tables/f1_outliers.csv", index=False)
mostrar("**Decisión.** Los valores extremos **no se eliminan**: corresponden a municipios reales grandes (p. ej. Amazonía y Orinoquía) que son justamente los de mayor interés para el sistema de alerta. "
        "En su lugar se aplica `log(1+x)`, que reduce de forma marcada el porcentaje de atípicos (columna *log1p*) sin descartar información.")

In [ ]:
cols_corr = ["forest_loss_ha", "area_km2", "population", "cattle_heads", "planted_area_ha", "nbi_pct", "coca_ha", "armed_groups"]
corr = df_log[cols_corr].corr()

fig, ax = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={"width_ratios": [1.1, 1]})
im = ax[0].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax[0].set_xticks(range(len(cols_corr))); ax[0].set_xticklabels(cols_corr, rotation=45, ha="right")
ax[0].set_yticks(range(len(cols_corr))); ax[0].set_yticklabels(cols_corr)
for i in range(len(cols_corr)):
    for j in range(len(cols_corr)):
        ax[0].text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8,
                   color="white" if abs(corr.iloc[i, j]) > 0.6 else "black")
ax[0].grid(False); ax[0].set_title("Correlación de Pearson (variables en log1p)")
plt.colorbar(im, ax=ax[0], fraction=0.046)

tend = df.groupby(["year", "region"])["forest_loss_ha"].sum().unstack()
for r in tend.columns:
    ax[1].plot(tend.index, tend[r] / 1000, marker="o", label=r)
ax[1].set_title("Pérdida total por región (miles de ha)"); ax[1].set_xlabel("año"); ax[1].legend(fontsize=8)
savefig("fig02_correlacion_y_tendencia")

top_corr = corr["forest_loss_ha"].drop("forest_loss_ha").sort_values(key=abs, ascending=False)
mostrar("**Correlación de cada variable con la pérdida (log):** " + ", ".join(f"`{k}` {v:.2f}" for k, v in top_corr.items()) +
        ". Las variables de tamaño (área, población, ganado, área sembrada) están correlacionadas entre sí, es decir, hay **multicolinealidad**: "
        "es una motivación concreta para probar **PCA** en la Fase 2.")

## 1.6 Viabilidad técnica y organizacional, limitaciones y riesgos

| Aspecto | Evaluación |
|---|---|
| **Técnica** | Viable: ~1.100 observaciones etiquetadas, 10 predictores, modelos clásicos que entrenan en segundos; despliegue en Flask + Render sin GPU |
| **Organizacional** | Viable: equipo de 3 personas, código en GitHub, aplicación ya publicada; el uso real exigiría datos oficiales y un responsable institucional |
| **Suficiencia de datos** | Suficiente para demostrar el método (188 municipios × 9 años). **Insuficiente para decisiones reales** por ser simulado |
| **Limitación principal** | Los datos son simulados: las métricas miden qué tan bien el método recupera la estructura simulada, no la deforestación real |
| **Riesgo 1** | Interpretar el pronóstico como cifra oficial → mitigación: etiquetado "simulado/ilustrativo" en la app |
| **Riesgo 2** | Sesgo hacia subestimar municipios con pérdidas extremas → se mide en la Fase 4 |
| **Riesgo 3** | Deriva de datos si cambian las condiciones (nuevos frentes de deforestación) → plan de monitoreo en la Fase 6 |
| **Riesgo 4** | Fuga de información temporal (usar el futuro para predecir el pasado) → validación estrictamente temporal |

## 1.7 Criterios de éxito medibles
Se fijan **antes de ver resultados** en el conjunto de prueba:

In [ ]:
SUCCESS = {
    "C1": "El MAE en prueba (2023) debe ser MENOR O IGUAL que el del referente ingenuo 'igual al año pasado'",
    "C2": "El R² (escala log) en prueba debe ser ≥ 0,60",
    "C3": "El intervalo de incertidumbre (±0,67σ, nominal 50 %) debe tener cobertura empírica entre 40 % y 60 % en prueba",
}
display(pd.DataFrame({"Criterio": SUCCESS.keys(), "Definición": SUCCESS.values()}))
RESULTS["criterios_exito"] = SUCCESS

---
# FASE 2 · Ingeniería de datos

## 2.1 Selección y justificación de fuentes y variables

* **Fuente:** `forest_loss_municipalities.csv` (simulado, inspirado en datos municipales públicos). Es la única fuente porque la aplicación solo recibe datos que un usuario puede ingresar a mano.
* **Variables históricas (rezagos):** pérdida de los **tres últimos años** (`lag0`, `lag1`, `lag2`). La deforestación es persistente: los frentes activos tienden a seguir activos.
* **Condiciones actuales (7):** área, población, ganado, área sembrada, NBI, coca y grupos armados; son los motores documentados de la deforestación (ganadería extensiva, cultivos ilícitos, presión social y control territorial).
* **Objetivo:** pérdida del **año siguiente** (`target`).

## 2.2 Construcción del panel con rezagos
Para cada municipio y año de origen *t* se arma una fila con `lag0 = pérdida(t)`, `lag1 = pérdida(t−1)`, `lag2 = pérdida(t−2)` y `target = pérdida(t+1)`.
Las filas sin `target` (origen 2023) quedan **sin etiqueta** y no se usan para evaluar.

In [ ]:
def construir_panel(frame):
    d = frame.sort_values(["municipality", "department", "year"]).copy()
    g = d.groupby(["municipality", "department"])["forest_loss_ha"]
    d["lag0"] = d["forest_loss_ha"]      # pérdida del último año conocido
    d["lag1"] = g.shift(1)               # hace 2 años
    d["lag2"] = g.shift(2)               # hace 3 años
    d["target"] = g.shift(-1)            # pérdida del año siguiente (lo que se predice)
    d["target_year"] = d["year"] + 1
    return d.dropna(subset=["lag1", "lag2"])

panel = construir_panel(df)
labeled = panel.dropna(subset=["target"]).reset_index(drop=True)

FEATURES = ["lag0", "lag1", "lag2", "area_km2", "population", "cattle_heads", "planted_area_ha", "nbi_pct", "coca_ha", "armed_groups"]
LOG_COLS = ["lag0", "lag1", "lag2", "area_km2", "population", "cattle_heads", "planted_area_ha", "coca_ha"]   # muy asimétricas → log(1+x)
PASS_COLS = ["nbi_pct", "armed_groups"]                                                                         # ya están en escala acotada
ORDERED = LOG_COLS + PASS_COLS

print("Panel con rezagos:", panel.shape, "| filas etiquetadas:", labeled.shape)
print("Años objetivo disponibles:", sorted(labeled.target_year.unique().tolist()))
display(labeled[["municipality", "year", "lag2", "lag1", "lag0", "target", "target_year"]].head(6))

## 2.3 Evidencia antes y después del procesamiento
Se comparan las distribuciones y la asimetría de las variables más sesgadas **antes** (escala original) y **después** de `log(1+x)`.

In [ ]:
vars_plot = ["cattle_heads", "coca_ha", "population", "lag0"]
etiquetas = {"cattle_heads": "Ganado (cabezas)", "coca_ha": "Coca (ha)", "population": "Población", "lag0": "Pérdida último año (ha)"}
fig, ax = plt.subplots(2, 4, figsize=(15, 6))
for j, v in enumerate(vars_plot):
    ax[0, j].hist(labeled[v], bins=50, color=GRIS);          ax[0, j].set_title(f"ANTES · {etiquetas[v]}", fontsize=9)
    ax[1, j].hist(np.log1p(labeled[v]), bins=50, color=VERDE); ax[1, j].set_title(f"DESPUÉS · log(1+x)", fontsize=9)
savefig("fig04_antes_despues_transformacion")

antes_despues = pd.DataFrame({
    "variable": LOG_COLS,
    "asimetría ANTES": [round(float(stats.skew(labeled[c])), 2) for c in LOG_COLS],
    "asimetría DESPUÉS (log1p)": [round(float(stats.skew(np.log1p(labeled[c]))), 2) for c in LOG_COLS],
})
display(antes_despues)
antes_despues.to_csv("reports/tables/f2_asimetria_antes_despues.csv", index=False)
RESULTS["asimetria_antes_despues"] = antes_despues.to_dict("records")

## 2.4 Partición temporal y validación (evita fuga de datos)
Como el problema es **pronosticar el futuro**, las particiones deben respetar el orden del tiempo (una partición aleatoria permitiría “ver el futuro” al entrenar):

* **Entrenamiento/validación (dev):** años objetivo **2018-2022**. Aquí se hace toda la selección y el ajuste.
* **Validación cruzada temporal con ventana creciente (3 pliegues):** se entrena con los años anteriores y se valida en 2020, luego 2021 y luego 2022.
* **Prueba (test):** año objetivo **2023**. **No se toca** hasta el final, una sola vez, con el modelo ya elegido.

In [ ]:
TEST_YEAR = 2023
FOLD_YEARS = (2020, 2021, 2022)

dev_df = labeled[labeled.target_year < TEST_YEAR].reset_index(drop=True)
test_df = labeled[labeled.target_year == TEST_YEAR].reset_index(drop=True)

X_dev, X_test = dev_df[FEATURES], test_df[FEATURES]
y_dev, y_test = np.log1p(dev_df["target"].to_numpy(float)), np.log1p(test_df["target"].to_numpy(float))
ha_dev, ha_test = dev_df["target"].to_numpy(float), test_df["target"].to_numpy(float)

folds = [(np.where(dev_df.target_year < fy)[0], np.where(dev_df.target_year == fy)[0]) for fy in FOLD_YEARS]

tabla_part = pd.DataFrame(
    [[f"Pliegue {i+1}", f"{dev_df.target_year[tr].min()}-{dev_df.target_year[tr].max()}", len(tr), fy, len(va)]
     for i, ((tr, va), fy) in enumerate(zip(folds, FOLD_YEARS))] +
    [["PRUEBA (final)", f"{dev_df.target_year.min()}-{dev_df.target_year.max()}", len(dev_df), TEST_YEAR, len(test_df)]],
    columns=["Etapa", "Años de entrenamiento", "Filas entrenamiento", "Año evaluado", "Filas evaluadas"])
display(tabla_part)
tabla_part.to_csv("reports/tables/f2_particiones.csv", index=False)
RESULTS["particion"] = {"dev_filas": int(len(dev_df)), "test_filas": int(len(test_df)), "test_year": TEST_YEAR, "folds": list(FOLD_YEARS)}

fig, ax = plt.subplots(figsize=(10, 2.6))
anios = list(range(2018, 2024))
for i, fy in enumerate(FOLD_YEARS):
    for a in anios:
        color = VERDE if a < fy else (DORADO if a == fy else "#E3E8E5")
        ax.barh(i, 0.9, left=a, color=color, edgecolor="white")
for a in anios:
    ax.barh(len(FOLD_YEARS), 0.9, left=a, color=VERDE if a < TEST_YEAR else ROJO, edgecolor="white")
ax.set_yticks(range(len(FOLD_YEARS) + 1)); ax.set_yticklabels([f"Pliegue {i+1}" for i in range(3)] + ["Prueba final"])
ax.set_xlabel("año objetivo"); ax.set_title("Verde = entrenamiento · Dorado = validación · Rojo = prueba final (se usa una sola vez)")
ax.invert_yaxis(); ax.grid(False)
savefig("fig05_particion_temporal")

## 2.5 Transformaciones y su justificación

| Paso | Variables | Justificación |
|---|---|---|
| `log(1+x)` | rezagos, área, población, ganado, área sembrada, coca | Corrige asimetría fuerte y valores extremos sin eliminarlos; `1+x` permite ceros |
| Se mantiene tal cual | `nbi_pct` (0-100) y `armed_groups` (0-5) | Ya tienen escala acotada |
| Estandarización (`StandardScaler`) | todos los predictores transformados | Necesaria para PCA y ayuda a la regresión lineal; los árboles no la necesitan pero no la perjudica |
| Transformación del objetivo `log(1+y)` | `target` | Los modelos aprenden en escala log; las métricas de hectáreas se calculan **devolviendo** la predicción a escala original con `expm1` |
| Codificación | — | No hay variables categóricas entre los predictores (la región se usa solo para analizar sesgo, no como entrada) |

**Anti-fuga de datos.** Todo el preprocesamiento vive **dentro de un `Pipeline` de scikit-learn**. Al validar, `fit` se ejecuta solo con las filas de entrenamiento de cada pliegue y el pliegue de validación solo se *transforma*. Abajo se demuestra:

In [ ]:
def construir_preprocesador():
    return Pipeline([
        ("columnas", ColumnTransformer([("log", FunctionTransformer(np.log1p), LOG_COLS),
                                        ("directo", "passthrough", PASS_COLS)])),
        ("escalado", StandardScaler()),
    ])

# Demostración: el escalador se ajusta SOLO con entrenamiento (pliegue 1) y no con la validación
tr, va = folds[0]
prep_demo = construir_preprocesador().fit(X_dev.iloc[tr])
media_train = prep_demo.named_steps["escalado"].mean_[:3]
media_todo = construir_preprocesador().fit(X_dev).named_steps["escalado"].mean_[:3]
display(pd.DataFrame({"variable": LOG_COLS[:3], "media aprendida SOLO con entrenamiento": media_train.round(4),
                      "media si se usara TODO (fuga)": media_todo.round(4)}))
mostrar("Las medias difieren: si se ajustara el escalador con todos los datos, la validación influiría en el preprocesamiento (**fuga de datos**). "
        "Con el `Pipeline` + validación temporal esto no ocurre.")

## 2.6 Reducción de dimensionalidad: PCA y t-SNE

Los 10 predictores no son independientes (sección 1.5 mostró multicolinealidad entre las variables de tamaño). La reducción de dimensionalidad permite (a) **entender** la estructura de los datos, (b) **visualizarla** en 2D y (c) **probar** si comprimir los predictores mejora o empeora el modelo.

### PCA — Análisis de Componentes Principales
* **Idea:** proyección **lineal** de los datos sobre ejes ortogonales (componentes) ordenados por la varianza que explican.
* **Cómo funciona:** con las variables estandarizadas $Z$, se diagonaliza la matriz de covarianza $C=\frac{1}{n-1}Z^\top Z$. Los vectores propios son las direcciones (componentes) y los valores propios $\lambda_k$ son la varianza de cada una. La proporción de varianza explicada es $\lambda_k/\sum_j \lambda_j$.
* **Decisión clave:** cuántos componentes conservar → se usa el **umbral del 95 % de varianza acumulada** (y se contrasta con el codo del *scree plot*).
* **Interpretación:** las **cargas (loadings)** indican qué variables originales pesan en cada componente.
* **Límites:** es lineal, no supervisado (ignora el objetivo) y los componentes son menos interpretables que las variables originales.

### t-SNE — *t-distributed Stochastic Neighbor Embedding*
* **Idea:** incrustación **no lineal** pensada para **visualizar** estructura local en 2D.
* **Cómo funciona:** convierte distancias en probabilidades de vecindad (gaussianas en alta dimensión, distribución *t* de Student en 2D) y minimiza la divergencia de Kullback-Leibler $KL(P\|Q)$ entre ambas.
* **Hiperparámetro principal:** `perplexity` ≈ número efectivo de vecinos. Valores bajos enfatizan estructura muy local; altos, más global.
* **Cuidados:** el tamaño de los grupos y las distancias *entre* grupos **no** son interpretables; es estocástico y **no transforma datos nuevos** (no tiene `transform`), por eso **solo se usa para exploración**, nunca como entrada del modelo desplegado.

> Todo el análisis exploratorio de PCA/t-SNE usa **únicamente datos de entrenamiento/validación (dev)**; el conjunto de prueba 2023 no interviene.

In [ ]:
prep_expl = construir_preprocesador().fit(X_dev)
Z_dev = prep_expl.transform(X_dev)

pca_full = PCA(random_state=SEED).fit(Z_dev)
evr = pca_full.explained_variance_ratio_
cum = np.cumsum(evr)
n95 = int(np.argmax(cum >= 0.95)) + 1
n80 = int(np.argmax(cum >= 0.80)) + 1

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(range(1, len(evr) + 1), evr * 100, color=VERDE)
ax[0].set_xlabel("componente principal"); ax[0].set_ylabel("% de varianza explicada"); ax[0].set_title("Scree plot (codo)")
ax[1].plot(range(1, len(cum) + 1), cum * 100, marker="o", color=VERDE_OSC)
ax[1].axhline(95, ls="--", color=ROJO); ax[1].axvline(n95, ls=":", color=ROJO)
ax[1].text(n95 + 0.15, 60, f"{n95} componentes\n≥ 95 % de varianza", color=ROJO)
ax[1].set_xlabel("número de componentes"); ax[1].set_ylabel("% de varianza acumulada"); ax[1].set_title("Varianza acumulada")
savefig("fig06_pca_varianza")

tabla_pca = pd.DataFrame({"componente": [f"PC{i+1}" for i in range(len(evr))],
                          "varianza explicada (%)": (evr * 100).round(2), "acumulada (%)": (cum * 100).round(2)})
display(tabla_pca)
tabla_pca.to_csv("reports/tables/f2_pca_varianza.csv", index=False)
RESULTS["pca"] = {"n_variables": len(FEATURES), "n95": n95, "n80": n80, "varianza_pc1": round(float(evr[0] * 100), 1),
                  "varianza_pc2": round(float(evr[1] * 100), 1), "acum_n95": round(float(cum[n95 - 1] * 100), 1)}
mostrar(f"**Resultado.** PC1 explica **{evr[0]*100:.1f} %** y PC2 **{evr[1]*100:.1f} %** de la varianza. "
        f"Se necesitan **{n80} componentes para el 80 %** y **{n95} para el 95 %** de los {len(FEATURES)} originales: "
        f"la reducción es de {len(FEATURES)} → {n95} dimensiones conservando el 95 % de la información lineal.")

In [ ]:
k = 4
carga = pd.DataFrame(pca_full.components_[:k], columns=ORDERED, index=[f"PC{i+1}" for i in range(k)])

fig, ax = plt.subplots(figsize=(11, 3.6))
im = ax.imshow(carga.values, cmap="RdBu_r", vmin=-0.8, vmax=0.8, aspect="auto")
ax.set_xticks(range(len(ORDERED))); ax.set_xticklabels(ORDERED, rotation=45, ha="right")
ax.set_yticks(range(k)); ax.set_yticklabels(carga.index)
for i in range(k):
    for j in range(len(ORDERED)):
        ax.text(j, i, f"{carga.values[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(carga.values[i, j]) > 0.45 else "black")
ax.grid(False); ax.set_title("Cargas (loadings) de los primeros componentes"); plt.colorbar(im, ax=ax, fraction=0.025)
savefig("fig07_pca_cargas")
carga.round(3).to_csv("reports/tables/f2_pca_cargas.csv")

lineas = []
for i in range(3):
    top = carga.iloc[i].abs().sort_values(ascending=False).head(3).index.tolist()
    lineas.append(f"* **PC{i+1}** ({evr[i]*100:.1f} %): pesa sobre todo en `{top[0]}`, `{top[1]}` y `{top[2]}`.")
mostrar("**Interpretación de los componentes**\n\n" + "\n".join(lineas))

In [ ]:
scores = pca_full.transform(Z_dev)[:, :2]
fig, ax = plt.subplots(figsize=(7.5, 5.5))
sc = ax.scatter(scores[:, 0], scores[:, 1], c=y_dev, cmap="YlGn", s=14, alpha=0.8)
escala = 3.2
for j, nombre in enumerate(ORDERED):
    ax.arrow(0, 0, pca_full.components_[0, j] * escala, pca_full.components_[1, j] * escala,
             color=ROJO, alpha=0.8, head_width=0.06)
    ax.text(pca_full.components_[0, j] * escala * 1.12, pca_full.components_[1, j] * escala * 1.12, nombre, fontsize=8, color=ROJO)
ax.set_xlabel(f"PC1 ({evr[0]*100:.1f} %)"); ax.set_ylabel(f"PC2 ({evr[1]*100:.1f} %)")
ax.set_title("Biplot PCA · color = log(1 + pérdida del año siguiente)")
plt.colorbar(sc, label="log(1 + ha)")
savefig("fig08_pca_biplot")

In [ ]:
perplexities = (5, 30, 50)
regiones = sorted(dev_df.region.unique())
paleta = dict(zip(regiones, ["#0B6B3A", "#C8A22A", "#B5412F", "#2F6DB5", "#7A4FA3"]))

fig, ax = plt.subplots(2, 3, figsize=(15, 8.5))
filas_tsne = []
for j, perp in enumerate(perplexities):
    t0 = time.time()
    emb = TSNE(n_components=2, perplexity=perp, init="pca", learning_rate="auto", random_state=SEED).fit_transform(Z_dev)
    tw = trustworthiness(Z_dev, emb, n_neighbors=10)
    filas_tsne.append([perp, round(float(tw), 4), round(time.time() - t0, 1)])
    sc = ax[0, j].scatter(emb[:, 0], emb[:, 1], c=y_dev, cmap="YlGn", s=12)
    ax[0, j].set_title(f"t-SNE · perplexity = {perp} (color = log pérdida)")
    for r in regiones:
        m = (dev_df.region == r).to_numpy()
        ax[1, j].scatter(emb[m, 0], emb[m, 1], s=12, color=paleta[r], label=r)
    ax[1, j].set_title(f"t-SNE · perplexity = {perp} (color = región)")
    if j == 0:
        ax[1, j].legend(fontsize=8)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
savefig("fig09_tsne")

tsne_tab = pd.DataFrame(filas_tsne, columns=["perplexity", "trustworthiness (10 vecinos)", "tiempo (s)"])
display(tsne_tab)
tsne_tab.to_csv("reports/tables/f2_tsne.csv", index=False)
mejor_p = int(tsne_tab.sort_values("trustworthiness (10 vecinos)", ascending=False).iloc[0]["perplexity"])
RESULTS["tsne"] = {"perplexities": list(perplexities), "trustworthiness": tsne_tab["trustworthiness (10 vecinos)"].tolist(), "mejor_perplexity": mejor_p}
mostrar(f"**Lectura.** La *trustworthiness* (0-1) mide qué tan bien se conservan los vecinos locales al pasar a 2D; fue más alta con `perplexity = {mejor_p}`. "
        "Se observa si municipios de la misma región y con pérdida parecida quedan cerca. t-SNE sirve para **explorar**; no se usa como entrada del modelo porque no puede transformar municipios nuevos.")

---
# FASE 3 · Ingeniería del modelo

## 3.1 Algoritmos seleccionados y justificación

| Modelo | Por qué se elige | Qué aporta |
|---|---|---|
| **Referentes ingenuos** (igual al año pasado, promedio de 3 años, mediana) | Un modelo solo es útil si supera estas reglas simples | Piso de comparación |
| **Regresión lineal múltiple** | Línea base interpretable; funciona bien con la persistencia de la pérdida en escala log | Coeficientes interpretables |
| **Random Forest** | Captura no linealidades e interacciones (p. ej. coca × grupos armados); robusto a valores extremos | Poco ajuste manual, importancia de variables |
| **Gradient Boosting** | Suele dar la mejor precisión en datos tabulares pequeños; corrige errores de forma secuencial | Alta capacidad, sensible a hiperparámetros |

**Conocimiento del dominio que guió las decisiones**
* La deforestación es **persistente** (frentes que siguen activos) → se incluyen tres rezagos.
* La pérdida es **multiplicativa y muy asimétrica** → se modela en escala logarítmica y se evalúa también en hectáreas.
* Se quiere **pronosticar el futuro** → validación temporal, nunca aleatoria.
* Las variables de tamaño están correlacionadas → se prueba PCA como alternativa.

## 3.2 Entrenamiento, validación y ajuste de hiperparámetros
Cada modelo se entrena **con y sin PCA (95 % de varianza)** y se ajusta con `GridSearchCV` usando los **3 pliegues temporales**. El PCA vive dentro del `Pipeline`, así que se ajusta solo con los datos de entrenamiento de cada pliegue. Métrica de selección: **MAE en hectáreas** (promedio de los 3 pliegues).

In [ ]:
def mae_hectareas(y_true_log, y_pred_log):
    # MAE en hectáreas: se devuelve la predicción de la escala log a la escala original
    return mean_absolute_error(np.expm1(y_true_log), np.clip(np.expm1(y_pred_log), 0, None))
mae_scorer = make_scorer(mae_hectareas, greater_is_better=False)

def metricas(ha_real, pred_log):
    pred = np.clip(np.expm1(pred_log), 0, None)
    return {"mae": float(mean_absolute_error(ha_real, pred)),
            "rmse": float(math.sqrt(np.mean((ha_real - pred) ** 2))),
            "r2_log": float(r2_score(np.log1p(ha_real), pred_log)),
            "r2_ha": float(r2_score(ha_real, pred))}

def pipeline_base(modelo, reductor="passthrough"):
    return Pipeline([("prep", construir_preprocesador()), ("reductor", reductor), ("modelo", modelo)])

MODELOS = {
    "Regresión lineal": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=SEED, n_jobs=1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=SEED),
}
REJILLAS = {
    "Regresión lineal": {},
    "Random Forest": {"modelo__n_estimators": [150, 300], "modelo__min_samples_leaf": [1, 3, 5], "modelo__max_depth": [None, 10]},
    "Gradient Boosting": {"modelo__n_estimators": [150, 300], "modelo__learning_rate": [0.03, 0.05, 0.1],
                          "modelo__max_depth": [2, 3], "modelo__subsample": [0.8]},
}
REDUCTORES = {"Sin PCA": "passthrough", "Con PCA (95 %)": PCA(n_components=0.95, random_state=SEED)}

# ---- Referentes ingenuos en los mismos pliegues ----
def mae_baselines_fold(tr, va):
    real = ha_dev[va]
    return {
        "Igual al año pasado": mean_absolute_error(real, dev_df["lag0"].to_numpy(float)[va]),
        "Promedio de los últimos 3 años": mean_absolute_error(real, dev_df[["lag0", "lag1", "lag2"]].mean(axis=1).to_numpy()[va]),
        "Siempre la mediana": mean_absolute_error(real, np.full(len(va), np.median(ha_dev[tr]))),
    }
base_folds = pd.DataFrame([mae_baselines_fold(tr, va) for tr, va in folds], index=list(FOLD_YEARS))
BASELINES_CV = base_folds.mean().to_dict()
display(base_folds.round(1).T.assign(media=base_folds.mean().round(1)))

# ---- Búsqueda de hiperparámetros ----
BUSQUEDAS, FILAS = {}, []
for nombre, modelo in MODELOS.items():
    for etiqueta, reductor in REDUCTORES.items():
        t0 = time.time()
        rejilla = {"reductor": [reductor], **REJILLAS[nombre]}
        gs = GridSearchCV(pipeline_base(modelo), rejilla, scoring=mae_scorer, cv=folds,
                          return_train_score=True, n_jobs=-1, refit=True)
        gs.fit(X_dev, y_dev)
        i, r = gs.best_index_, gs.cv_results_
        por_pliegue = [-r[f"split{k}_test_score"][i] for k in range(len(folds))]
        red_ajustado = gs.best_estimator_.named_steps["reductor"]
        BUSQUEDAS[(nombre, etiqueta)] = gs
        FILAS.append({
            "Modelo": nombre, "Reducción": etiqueta,
            "Mejores parámetros": ", ".join(f"{k.replace('modelo__', '')}={v}" for k, v in gs.best_params_.items() if k != "reductor") or "—",
            "MAE CV (ha)": round(-gs.best_score_, 1), "Desv. entre pliegues": round(float(np.std(por_pliegue)), 1),
            **{f"MAE {fy}": round(v, 1) for fy, v in zip(FOLD_YEARS, por_pliegue)},
            "MAE entrenamiento (ha)": round(-r["mean_train_score"][i], 1),
            "Componentes": int(red_ajustado.n_components_) if hasattr(red_ajustado, "n_components_") else len(FEATURES),
            "Combinaciones": len(r["params"]), "Tiempo búsqueda (s)": round(time.time() - t0, 1),
        })
        print(f"{nombre:18s} | {etiqueta:15s} | MAE CV = {-gs.best_score_:7.1f} ha | {len(r['params'])} combinaciones | {time.time()-t0:5.1f}s")

EXP = pd.DataFrame(FILAS)
EXP["Brecha CV − entrenamiento"] = (EXP["MAE CV (ha)"] - EXP["MAE entrenamiento (ha)"]).round(1)

## 3.3 Registro de experimentos
Cada fila es un experimento (modelo × reducción de dimensionalidad) con su mejor configuración tras la búsqueda en rejilla.

In [ ]:
display(EXP)
EXP.to_csv("reports/tables/f3_experimentos.csv", index=False)
pd.DataFrame({"Referente": list(BASELINES_CV), "MAE CV (ha)": [round(v, 1) for v in BASELINES_CV.values()]}).to_csv("reports/tables/f3_referentes_cv.csv", index=False)

total_comb = int(EXP["Combinaciones"].sum()); total_fits = total_comb * len(folds)
RESULTS["experimentos"] = {"n_experimentos": int(len(EXP)), "combinaciones": total_comb, "ajustes_totales": total_fits}
mostrar(f"**Se evaluaron {len(EXP)} configuraciones finales ({total_comb} combinaciones de hiperparámetros × {len(folds)} pliegues = {total_fits} ajustes).** "
        "Todos los parámetros, el valor de MAE por pliegue y la brecha entre entrenamiento y validación quedan registrados en `reports/tables/f3_experimentos.csv`.")

---
# FASE 4 · Evaluación del modelo

## 4.1 Métricas y justificación

| Métrica | Fórmula / escala | Por qué se usa |
|---|---|---|
| **MAE** (ha) | $\frac{1}{n}\sum \lvert y-\hat y\rvert$ | Error promedio **en hectáreas**: fácil de explicar a autoridades; es el criterio de selección |
| **RMSE** (ha) | $\sqrt{\frac{1}{n}\sum (y-\hat y)^2}$ | Penaliza más los errores grandes, que ocurren en los municipios amazónicos con mayor pérdida |
| **R² (escala log)** | $1-\frac{SS_{res}}{SS_{tot}}$ sobre $\log(1+y)$ | Mide el ajuste **relativo** en todo el rango (de 0 a más de 10.000 ha) |
| **R² (ha)** | igual, en hectáreas | Complementario: dominado por los pocos municipios extremos |

**Validación cruzada adecuada a la estructura de los datos:** ventana creciente por año (3 pliegues). Una validación aleatoria (k-fold) mezclaría años y filtraría información del futuro.

In [ ]:
orden = EXP.sort_values("MAE CV (ha)").reset_index(drop=True)
mejor_fila = orden.iloc[0]
MEJOR = (mejor_fila["Modelo"], mejor_fila["Reducción"])
mejor_pipe = BUSQUEDAS[MEJOR].best_estimator_

etiquetas = [f"{m}\n{r}" for m, r in zip(EXP["Modelo"], EXP["Reducción"])]
fig, ax = plt.subplots(1, 2, figsize=(15, 4.6))
colores = [VERDE if r == "Sin PCA" else DORADO for r in EXP["Reducción"]]
ax[0].bar(range(len(EXP)), EXP["MAE CV (ha)"], yerr=EXP["Desv. entre pliegues"], color=colores, capsize=4)
ax[0].set_xticks(range(len(EXP))); ax[0].set_xticklabels(etiquetas, fontsize=8)
estilos = {"Igual al año pasado": "--", "Promedio de los últimos 3 años": ":", "Siempre la mediana": "-."}
for (nombre, v), ls in zip(BASELINES_CV.items(), estilos.values()):
    ax[0].axhline(v, ls=ls, color=ROJO if "pasado" in nombre else GRIS, label=f"Referente: {nombre} ({v:.0f})")
ax[0].set_ylabel("MAE en validación cruzada (ha)"); ax[0].set_title("Comparación de modelos (menor es mejor) · barras de error = desv. entre pliegues")
ax[0].legend(fontsize=7, loc="upper right")
for k, fy in enumerate(FOLD_YEARS):
    ax[1].plot(range(len(EXP)), EXP[f"MAE {fy}"], marker="o", label=f"Validación {fy}")
ax[1].set_xticks(range(len(EXP))); ax[1].set_xticklabels(etiquetas, fontsize=8); ax[1].legend()
ax[1].set_ylabel("MAE (ha)"); ax[1].set_title("MAE por pliegue (estabilidad temporal)")
savefig("fig10_comparacion_cv")

display(orden[["Modelo", "Reducción", "MAE CV (ha)", "Desv. entre pliegues", "MAE entrenamiento (ha)", "Brecha CV − entrenamiento", "Componentes"]])
mostrar(f"### Modelo seleccionado (criterio: menor MAE promedio en validación cruzada temporal)\n\n"
        f"**{MEJOR[0]} · {MEJOR[1]}** — MAE CV = **{mejor_fila['MAE CV (ha)']:.1f} ha** (±{mejor_fila['Desv. entre pliegues']:.1f}). "
        f"Mejores parámetros: `{mejor_fila['Mejores parámetros']}`.")
RESULTS["modelo_seleccionado"] = {"modelo": MEJOR[0], "reduccion": MEJOR[1], "mae_cv": float(mejor_fila["MAE CV (ha)"]),
                                  "parametros": mejor_fila["Mejores parámetros"]}

## 4.2 Sobreajuste, subajuste y sesgo
Se compara el error en entrenamiento con el error en validación (brecha) y se construye una **curva de aprendizaje** variando cuántos años de historia se usan para entrenar (validando siempre en 2022).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
x = np.arange(len(EXP)); w = 0.38
ax[0].bar(x - w/2, EXP["MAE entrenamiento (ha)"], w, label="Entrenamiento", color=GRIS)
ax[0].bar(x + w/2, EXP["MAE CV (ha)"], w, label="Validación (CV)", color=VERDE)
ax[0].set_xticks(x); ax[0].set_xticklabels(etiquetas, fontsize=8); ax[0].legend()
ax[0].set_ylabel("MAE (ha)"); ax[0].set_title("Entrenamiento vs. validación (brecha = sobreajuste)")

lc = []
for k in (2018, 2019, 2020, 2021):
    tr = np.where(dev_df.target_year <= k)[0]; va = np.where(dev_df.target_year == 2022)[0]
    est = clone(mejor_pipe).fit(X_dev.iloc[tr], y_dev[tr])
    lc.append([len(tr), mae_hectareas(y_dev[tr], est.predict(X_dev.iloc[tr])), mae_hectareas(y_dev[va], est.predict(X_dev.iloc[va]))])
lc = pd.DataFrame(lc, columns=["filas de entrenamiento", "MAE entrenamiento", "MAE validación 2022"])
ax[1].plot(lc["filas de entrenamiento"], lc["MAE entrenamiento"], marker="o", color=GRIS, label="Entrenamiento")
ax[1].plot(lc["filas de entrenamiento"], lc["MAE validación 2022"], marker="o", color=VERDE, label="Validación 2022")
ax[1].set_xlabel("filas de entrenamiento (más años de historia →)"); ax[1].set_ylabel("MAE (ha)")
ax[1].set_title(f"Curva de aprendizaje · {MEJOR[0]} · {MEJOR[1]}"); ax[1].legend()
savefig("fig11_sobreajuste_curva_aprendizaje")
lc.round(1).to_csv("reports/tables/f4_curva_aprendizaje.csv", index=False)
display(lc.round(1))

brecha = mejor_fila["Brecha CV − entrenamiento"]; rel = brecha / max(mejor_fila["MAE entrenamiento (ha)"], 1e-9)
diag = "**sobreajuste marcado**" if rel > 0.5 else ("**sobreajuste moderado**" if rel > 0.2 else "**sobreajuste bajo**")
gap_lin = float(EXP[(EXP.Modelo == "Regresión lineal") & (EXP["Reducción"] == "Sin PCA")]["Brecha CV − entrenamiento"].iloc[0])
nota_arboles = (" En Random Forest y Gradient Boosting una brecha grande es **esperable**: cada árbol ve los datos de entrenamiento, así que el error *en entrenamiento* es optimista por construcción. "
                f"Por eso la selección se hace con el error de **validación temporal** y no con el de entrenamiento (la regresión lineal tiene una brecha de solo {gap_lin:.1f} ha, pero un error de validación mayor).") if MEJOR[0] != "Regresión lineal" else ""
v_prev, v_last = float(lc["MAE validación 2022"].iloc[-2]), float(lc["MAE validación 2022"].iloc[-1])
curva = (f"En la curva de aprendizaje el error de validación pasa de {lc['MAE validación 2022'].iloc[0]:.0f} ha (1 año de historia) a {v_last:.0f} ha (4 años): "
         + ("sigue bajando al añadir años, así que **más historia probablemente seguiría mejorando el modelo**." if v_last < v_prev * 0.98 else "ya se estabiliza, así que **más historia aportaría poco**; el límite lo ponen el ruido y la información disponible."))
mostrar(f"**Diagnóstico del modelo seleccionado.** MAE entrenamiento = {mejor_fila['MAE entrenamiento (ha)']:.1f} ha frente a {mejor_fila['MAE CV (ha)']:.1f} ha en validación: "
        f"brecha de {brecha:.1f} ha ({rel*100:.0f} % del error de entrenamiento) → {diag}." + nota_arboles + " " + curva)

## 4.3 Resultado final en el conjunto de prueba independiente (2023)
El modelo seleccionado ya fue reentrenado con **todos** los años de desarrollo (2018-2022). Ahora se evalúa **una sola vez** en 2023, año que no intervino en ninguna decisión.

In [ ]:
pred_test_log = mejor_pipe.predict(X_test)
M_TEST = metricas(ha_test, pred_test_log)

base_test = {
    "Igual al año pasado": test_df["lag0"].to_numpy(float),
    "Promedio de los últimos 3 años": test_df[["lag0", "lag1", "lag2"]].mean(axis=1).to_numpy(),
    "Siempre la mediana": np.full(len(test_df), np.median(ha_dev)),
}
filas = [[f"{MEJOR[0]} · {MEJOR[1]} (seleccionado)", *[M_TEST[k] for k in ("mae", "rmse", "r2_log", "r2_ha")]]]
BASE_TEST = {}
for nombre, pred in base_test.items():
    pred = np.asarray(pred, float)
    BASE_TEST[nombre] = {"mae": float(mean_absolute_error(ha_test, pred)), "rmse": float(math.sqrt(np.mean((ha_test - pred) ** 2))),
                         "r2_log": float(r2_score(np.log1p(ha_test), np.log1p(pred))), "r2_ha": float(r2_score(ha_test, pred))}
    filas.append([f"Referente: {nombre}", *[BASE_TEST[nombre][k] for k in ("mae", "rmse", "r2_log", "r2_ha")]])
tab_test = pd.DataFrame(filas, columns=["Modelo", "MAE (ha)", "RMSE (ha)", "R² (log)", "R² (ha)"]).round(3)
display(tab_test)
tab_test.to_csv("reports/tables/f4_resultado_prueba.csv", index=False)

# --- Incertidumbre: sigma a partir de residuos fuera de muestra de la validación cruzada ---
res_oof = []
for tr, va in folds:
    est = clone(mejor_pipe).fit(X_dev.iloc[tr], y_dev[tr])
    res_oof.extend((y_dev[va] - est.predict(X_dev.iloc[va])).tolist())
SIGMA = float(np.std(res_oof))
Z_INT = 0.67
bajo, alto = np.expm1(pred_test_log - Z_INT * SIGMA), np.expm1(pred_test_log + Z_INT * SIGMA)
cobertura = float(np.mean((ha_test >= bajo) & (ha_test <= alto)) * 100)

# --- Verificación de criterios de éxito (definidos en la Fase 1) ---
base_mae = BASE_TEST["Igual al año pasado"]["mae"]
verif = [
    ["C1", SUCCESS["C1"], f"{M_TEST['mae']:.1f} ha vs {base_mae:.1f} ha", "CUMPLE" if M_TEST["mae"] <= base_mae else "NO CUMPLE"],
    ["C2", SUCCESS["C2"], f"{M_TEST['r2_log']:.3f}", "CUMPLE" if M_TEST["r2_log"] >= 0.60 else "NO CUMPLE"],
    ["C3", SUCCESS["C3"], f"{cobertura:.1f} %", "CUMPLE" if 40 <= cobertura <= 60 else "NO CUMPLE"],
]
tab_crit = pd.DataFrame(verif, columns=["Criterio", "Definición", "Resultado en prueba 2023", "Veredicto"])
display(tab_crit)
tab_crit.to_csv("reports/tables/f4_criterios_exito.csv", index=False)

RESULTS["prueba"] = {"anio": TEST_YEAR, "n": int(len(test_df)), **{k: round(v, 3) for k, v in M_TEST.items()},
                     "referentes": {k: {m: round(x, 3) for m, x in v.items()} for k, v in BASE_TEST.items()},
                     "sigma": round(SIGMA, 4), "cobertura_intervalo": round(cobertura, 1),
                     "criterios": {r[0]: r[3] for r in verif}}
mejora = (1 - M_TEST["mae"] / base_mae) * 100
txt_mejora = f"**{mejora:.1f} % menor**" if mejora >= 0 else f"**{-mejora:.1f} % mayor**"
nota_modesta = (" La ventaja sobre el referente ingenuo es **modesta**: gran parte de la pérdida del próximo año ya se explica por la del año actual (persistencia), "
                "por eso el valor agregado del modelo está sobre todo en combinar historia y condiciones, en los niveles de alerta y en el rango de incertidumbre.") if abs(mejora) < 10 else ""
nota_r2 = (f" El R² (log) cumple el umbral por un margen **estrecho** ({M_TEST['r2_log'] - 0.60:+.3f}); conviene no sobreinterpretarlo.") if 0 <= M_TEST["r2_log"] - 0.60 < 0.03 else ""
mostrar(f"**Resultado en la prueba independiente.** En 2023 el modelo seleccionado obtiene MAE = **{M_TEST['mae']:.1f} ha**, RMSE = {M_TEST['rmse']:.1f} ha y R² (log) = {M_TEST['r2_log']:.3f}. "
        f"Frente al referente «igual al año pasado» ({base_mae:.1f} ha), su MAE es {txt_mejora}." + nota_modesta + nota_r2 +
        f" Criterios de éxito: C1 {verif[0][3]}, C2 {verif[1][3]}, C3 {verif[2][3]}.")

In [ ]:
pred_test = np.clip(np.expm1(pred_test_log), 0, None)
resid_log = y_test - pred_test_log

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
lim = [0, max(np.log1p(ha_test).max(), pred_test_log.max()) + 0.3]
ax[0].scatter(np.log1p(ha_test), pred_test_log, s=16, color=VERDE, alpha=0.75)
ax[0].plot(lim, lim, "--", color=ROJO); ax[0].set_xlabel("real  log(1+ha)"); ax[0].set_ylabel("predicho  log(1+ha)")
ax[0].set_title(f"Real vs. predicho · prueba {TEST_YEAR}")
ax[1].hist(resid_log, bins=30, color=DORADO); ax[1].axvline(0, color=ROJO, ls="--")
ax[1].set_xlabel("residuo (log)"); ax[1].set_title("Distribución de residuos")
ax[2].scatter(pred_test_log, resid_log, s=16, color=VERDE_OSC, alpha=0.7); ax[2].axhline(0, color=ROJO, ls="--")
ax[2].set_xlabel("predicho log(1+ha)"); ax[2].set_ylabel("residuo (log)"); ax[2].set_title("Residuos vs. predicho (¿hay patrón?)")
savefig("fig13_diagnostico_residuos")

# --- Sesgo por región y por nivel de pérdida ---
tmp = test_df[["municipality", "region"]].copy()
tmp["real"], tmp["pred"] = ha_test, pred_test
tmp["error"] = tmp["pred"] - tmp["real"]
por_region = tmp.groupby("region").agg(municipios=("real", "size"), MAE=("error", lambda s: s.abs().mean()), sesgo_medio=("error", "mean")).round(1)
tmp["nivel_real"] = pd.qcut(tmp["real"].rank(method="first"), 4, labels=["Q1 (menor pérdida)", "Q2", "Q3", "Q4 (mayor pérdida)"])
por_nivel = tmp.groupby("nivel_real", observed=True).agg(municipios=("real", "size"), MAE=("error", lambda s: s.abs().mean()), sesgo_medio=("error", "mean")).round(1)
display(por_region); display(por_nivel)
por_region.to_csv("reports/tables/f4_sesgo_region.csv"); por_nivel.to_csv("reports/tables/f4_sesgo_nivel.csv")
RESULTS["sesgo"] = {"por_region": por_region.reset_index().to_dict("records"), "por_nivel": por_nivel.reset_index().astype({"nivel_real": str}).to_dict("records")}

sub = por_nivel["sesgo_medio"].iloc[-1]
mostrar(f"**Sesgo.** En el cuartil de mayor pérdida real el error medio (predicho − real) es **{sub:+.1f} ha**: "
        + ("el modelo **subestima** los casos extremos, que son justamente los más críticos para el sistema de alerta; es el riesgo principal a comunicar." if sub < 0 else "no hay subestimación sistemática en los casos extremos.") +
        f" El intervalo ±{Z_INT}σ (σ = {SIGMA:.3f} en log) cubrió **{cobertura:.1f} %** de los valores reales de 2023 (nominal ≈ 50 %).")

In [ ]:
imp = permutation_importance(mejor_pipe, X_test, y_test, scoring=mae_scorer, n_repeats=20, random_state=SEED)
imp_df = pd.DataFrame({"variable": FEATURES, "aumento_MAE_ha": imp.importances_mean, "desv": imp.importances_std}).sort_values("aumento_MAE_ha")
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(imp_df["variable"], imp_df["aumento_MAE_ha"], xerr=imp_df["desv"], color=VERDE, capsize=3)
ax.set_xlabel("aumento del MAE (ha) al desordenar la variable"); ax.set_title("Importancia por permutación (descriptiva, sobre prueba 2023)")
savefig("fig15_importancia_variables")
imp_df.sort_values("aumento_MAE_ha", ascending=False).round(2).to_csv("reports/tables/f4_importancia.csv", index=False)
top3 = imp_df.sort_values("aumento_MAE_ha", ascending=False)["variable"].head(3).tolist()
RESULTS["importancia_top3"] = top3
mostrar(f"**Variables más influyentes:** `{top3[0]}`, `{top3[1]}` y `{top3[2]}`. "
        "La importancia se calcula *después* de elegir el modelo, solo para explicar; no se usó para seleccionar ni ajustar.")

## 4.4 ¿Cómo afecta la reducción de dimensionalidad (PCA) al modelo?
Comparación directa de cada algoritmo **con y sin PCA**, usando la validación cruzada temporal.

In [ ]:
piv = EXP.pivot(index="Modelo", columns="Reducción", values="MAE CV (ha)")
piv_t = EXP.pivot(index="Modelo", columns="Reducción", values="Tiempo búsqueda (s)")
comp = pd.DataFrame({
    "MAE sin PCA (ha)": piv["Sin PCA"], "MAE con PCA (ha)": piv["Con PCA (95 %)"],
    "Cambio por PCA (%)": ((piv["Con PCA (95 %)"] / piv["Sin PCA"] - 1) * 100).round(1),
    "Dimensiones (sin → con)": [f"{len(FEATURES)} → {int(EXP[(EXP.Modelo == m) & (EXP['Reducción'] == 'Con PCA (95 %)')]['Componentes'].iloc[0])}" for m in piv.index],
    "Tiempo búsqueda sin PCA (s)": piv_t["Sin PCA"], "Tiempo búsqueda con PCA (s)": piv_t["Con PCA (95 %)"],
})
display(comp)
comp.to_csv("reports/tables/f4_efecto_pca.csv")

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(comp)); w = 0.38
ax.bar(x - w/2, comp["MAE sin PCA (ha)"], w, label="Sin PCA", color=VERDE)
ax.bar(x + w/2, comp["MAE con PCA (ha)"], w, label="Con PCA (95 %)", color=DORADO)
ax.set_xticks(x); ax.set_xticklabels(comp.index); ax.set_ylabel("MAE CV (ha)"); ax.legend(); ax.set_title("Efecto de PCA en el error de validación")
savefig("fig14_efecto_pca")

RESULTS["efecto_pca"] = comp.reset_index().to_dict("records")
partes = []
for m, fila in comp.iterrows():
    c = fila["Cambio por PCA (%)"]
    partes.append(f"**{m}**: {c:+.1f} % ({'sin cambio apreciable' if abs(c) < 1 else ('mejora' if c < 0 else 'empeora')})")
mejoran = int((comp["Cambio por PCA (%)"] <= -1).sum()); empeoran = int((comp["Cambio por PCA (%)"] >= 1).sum())
mostrar(f"**Conclusión sobre PCA.** Al comprimir de {len(FEATURES)} a ~{n95} componentes el cambio del MAE de validación fue: " + "; ".join(partes) + ". "
        f"Es decir, PCA **mejoró de forma apreciable el MAE en {mejoran} de {len(comp)} modelos y lo empeoró en {empeoran}**. "
        "Una explicación plausible: PCA es *no supervisado* (ordena por varianza, no por utilidad para predecir), así que los componentes de poca varianza que se descartan pueden contener señal predictiva; "
        "además los árboles no sufren con la colinealidad, y en regresión lineal la colinealidad afecta sobre todo la interpretación de los coeficientes, no tanto la predicción. "
        "Además PCA vuelve el modelo menos interpretable (los componentes mezclan variables que un usuario no puede ingresar por separado). "
        "Por eso **PCA y t-SNE se usan aquí para comprender y visualizar los datos, y solo se incorporarían al modelo desplegado si mejoraran la validación, cosa que no ocurrió**.")

## 4.5 Revisión de relevancia práctica con una persona experta (**completar**)

> ⚠️ **Esta sección no puede generarla el código: debe hacerse con una persona real** (docente, profesional ambiental o de ciencia de datos). Presente la aplicación y los resultados, y registre aquí lo que realmente ocurrió.

| Campo | Registro |
|---|---|
| Fecha de la revisión | *(completar)* |
| Persona experta (nombre, cargo, institución) | *(completar)* |
| Qué se le mostró | *(p. ej. pronóstico, niveles de alerta, rango de incertidumbre, comparación de modelos)* |
| Comentarios sobre la utilidad práctica | *(completar)* |
| Críticas / riesgos que señaló | *(completar)* |
| Decisiones o cambios tomados a partir del feedback | *(completar)* |

**Preguntas sugeridas:** (1) ¿Los niveles de alerta Bajo/Moderado/Alto/Crítico son útiles para decidir? (2) ¿Un error medio de este tamaño es aceptable para priorizar municipios? (3) ¿Qué variables faltan que usted consideraría clave?

---
# FASE 5 · Despliegue

## 5.1 Artefactos guardados
Se guarda **un único archivo** con el *pipeline completo* (preprocesamiento + modelo), de modo que la aplicación nunca aplica transformaciones distintas de las del entrenamiento. También se guardan el sigma del intervalo de incertidumbre, los umbrales de alerta y los metadatos de versión.

> **Importante para Render:** los archivos `.joblib` dependen de la versión de scikit-learn con que se crearon. Fije en `requirements.txt` la **misma versión** que imprime la celda de preparación (por ejemplo `scikit-learn==X.Y.Z`) o vuelva a generar el modelo en el mismo entorno.

In [ ]:
QUANT = {"0.25": float(np.quantile(ha_dev, 0.25)), "0.5": float(np.quantile(ha_dev, 0.5)),
         "0.75": float(np.quantile(ha_dev, 0.75)), "0.95": float(np.quantile(ha_dev, 0.95))}

def nivel_alerta(estimado_ha):
    if estimado_ha < QUANT["0.25"]: return "Bajo"
    if estimado_ha < QUANT["0.75"]: return "Moderado"
    if estimado_ha < QUANT["0.95"]: return "Alto"
    return "Crítico"

LIMITES = {  # rangos válidos de entrada (de los datos de entrenamiento, con margen) para validar el formulario
    "loss_last_year": (0, 20000), "loss_2y_ago": (0, 20000), "loss_3y_ago": (0, 20000),
    "area_km2": (1, 100000), "population": (100, 10_000_000), "cattle_heads": (0, 10_000_000),
    "planted_area_ha": (0, 1_000_000), "nbi_pct": (0, 100), "coca_ha": (0, 20000), "armed_groups": (0, 10),
}

def predecir_perdida(artefacto, loss_last_year, loss_2y_ago, loss_3y_ago, area_km2, population,
                     cattle_heads, planted_area_ha, nbi_pct, coca_ha, armed_groups):
    # Pronóstico de pérdida de bosque del año siguiente para UN municipio. Lanza ValueError si hay entradas inválidas.
    entradas = dict(loss_last_year=loss_last_year, loss_2y_ago=loss_2y_ago, loss_3y_ago=loss_3y_ago, area_km2=area_km2,
                    population=population, cattle_heads=cattle_heads, planted_area_ha=planted_area_ha,
                    nbi_pct=nbi_pct, coca_ha=coca_ha, armed_groups=armed_groups)
    for k, v in entradas.items():
        lo, hi = LIMITES[k]
        if v is None or not np.isfinite(v) or not (lo <= v <= hi):
            raise ValueError(f"{k} fuera del rango válido [{lo}, {hi}]: {v}")
    fila = pd.DataFrame([{"lag0": loss_last_year, "lag1": loss_2y_ago, "lag2": loss_3y_ago, "area_km2": area_km2,
                          "population": population, "cattle_heads": cattle_heads, "planted_area_ha": planted_area_ha,
                          "nbi_pct": nbi_pct, "coca_ha": coca_ha, "armed_groups": armed_groups}])[artefacto["features"]]
    log_pred = float(artefacto["pipeline"].predict(fila)[0])
    est = max(float(np.expm1(log_pred)), 0.0)
    s, z = artefacto["sigma"], artefacto["z"]
    return {"estimado_ha": round(est), "rango_bajo_ha": max(round(float(np.expm1(log_pred - z * s))), 0),
            "rango_alto_ha": round(float(np.expm1(log_pred + z * s))), "nivel_alerta": nivel_alerta(est)}

ARTEFACTO = {"pipeline": mejor_pipe, "features": FEATURES, "sigma": SIGMA, "z": Z_INT, "quantiles": QUANT,
             "modelo": MEJOR[0], "reduccion": MEJOR[1], "entrenado_con": f"años objetivo {dev_df.target_year.min()}-{dev_df.target_year.max()} (dataset SIMULADO)",
             "sklearn": sklearn.__version__, "semilla": SEED}
RUTA_MODELO = "models/crispml_pipeline.joblib"
joblib.dump(ARTEFACTO, RUTA_MODELO, compress=3)
print(f"Guardado: {RUTA_MODELO} ({os.path.getsize(RUTA_MODELO)/1e6:.2f} MB)")

# Comprobación: recargar desde disco y verificar que da exactamente las mismas predicciones
cargado = joblib.load(RUTA_MODELO)
assert np.allclose(cargado["pipeline"].predict(X_test), pred_test_log), "El modelo recargado no coincide"
print("✔ El modelo recargado produce exactamente las mismas predicciones.")

# Demostración con un municipio real de la prueba
ej = test_df.sort_values("target", ascending=False).iloc[10]
salida = predecir_perdida(cargado, ej.lag0, ej.lag1, ej.lag2, ej.area_km2, ej.population, ej.cattle_heads,
                          ej.planted_area_ha, ej.nbi_pct, ej.coca_ha, ej.armed_groups)
print(f"\nEjemplo: {ej.municipality} ({ej.department}) · pérdida real 2023 = {ej.target:.0f} ha")
print(json.dumps(salida, indent=2, ensure_ascii=False))
RESULTS["despliegue"] = {"archivo": RUTA_MODELO, "tamano_mb": round(os.path.getsize(RUTA_MODELO) / 1e6, 2), "sklearn": sklearn.__version__,
                         "umbrales_alerta_ha": {k: round(v, 1) for k, v in QUANT.items()}}

## 5.2 Entradas, salidas y dependencias

**Entradas** (formulario `/predict` de la aplicación o llamada a `predecir_perdida`)

| Campo | Unidad | Rango válido | Descripción |
|---|---|---|---|
| `loss_last_year` | ha | 0 – 20.000 | Pérdida de bosque del último año conocido |
| `loss_2y_ago`, `loss_3y_ago` | ha | 0 – 20.000 | Pérdida de hace 2 y 3 años |
| `area_km2` | km² | 1 – 100.000 | Superficie del municipio |
| `population` | personas | 100 – 10.000.000 | Población |
| `cattle_heads` | cabezas | 0 – 10.000.000 | Ganado |
| `planted_area_ha` | ha | 0 – 1.000.000 | Área sembrada |
| `nbi_pct` | % | 0 – 100 | Necesidades básicas insatisfechas |
| `coca_ha` | ha | 0 – 20.000 | Cultivos de coca |
| `armed_groups` | conteo | 0 – 10 | Grupos armados presentes |

**Salidas:** `estimado_ha` (pérdida esperada del año siguiente), `rango_bajo_ha` / `rango_alto_ha` (intervalo ±0,67σ) y `nivel_alerta` (Bajo / Moderado / Alto / Crítico según cuantiles históricos 25 %, 75 % y 95 %).

**Dependencias:** Python 3.10+, `Flask`, `gunicorn`, `pandas`, `numpy`, `scikit-learn` (versión fijada), `joblib`, `scipy` (monitoreo). **Ejecución local:** `pip install -r requirements.txt` → `python app.py` → `http://127.0.0.1:5000`.

## 5.3 Instrucciones iniciales para el usuario
1. Ingrese a la aplicación e inicie sesión (o regístrese como ciudadano).
2. Abra **Predict**, escriba o seleccione los datos de su municipio y envíe el formulario.
3. Lea el **nivel de alerta**, el **estimado en hectáreas** y el **rango**: el rango indica cuánta incertidumbre hay (un rango ancho significa menos confianza).
4. Los usuarios con rol *Authority* pueden usar el **simulador *what-if*** para ver cómo cambiaría el pronóstico si, por ejemplo, disminuye el ganado o la coca.
5. Recuerde: **los datos del proyecto son simulados**; el resultado es ilustrativo y no reemplaza las cifras oficiales del IDEAM.

## 5.4 Plan de integración con el sistema de información de la Actividad 2
| Etapa | Acción |
|---|---|
| 1. Contrato de datos | Exponer un servicio `POST /api/predict` (JSON) con las mismas entradas/salidas anteriores y validación de rangos |
| 2. Autenticación | Reutilizar los roles *citizen / authority / admin*; el sistema de la Actividad 2 consume el servicio con credenciales de rol |
| 3. Almacenamiento | Guardar cada pronóstico en la tabla `forecasts` (usuario, municipio, estimado, nivel, fecha) para trazabilidad e historial |
| 4. Integración gradual | Primero lectura (consulta de pronósticos), luego escritura (alertas hacia el módulo de gestión) |
| 5. Pruebas | Pruebas de contrato con casos válidos e inválidos antes de pasar a producción |

## 5.5 Procedimientos de monitoreo, incidentes y retroalimentación
* **Monitoreo:** registro automático de cada predicción (fecha, entradas, salida, latencia) y revisión periódica de los indicadores de la Fase 6.
* **Reporte de incidentes:** toda entrada inválida o error de la aplicación se registra con marca de tiempo; un incidente se clasifica en *menor* (entrada rechazada), *mayor* (error del servicio) o *crítico* (pronósticos claramente erróneos). Los mayores y críticos se reportan al administrador por el canal acordado y se documentan con causa y acción correctiva.
* **Retroalimentación de usuarios:** formulario breve de utilidad (1-5) y comentarios asociado a cada pronóstico; se revisa junto con las métricas.

---
# FASE 6 · Monitoreo y mantenimiento

> ⚠️ **Todos los datos operativos de esta fase son SIMULADOS** (no hay usuarios reales todavía). Se generan a partir del conjunto de prueba con deriva artificial **solo para demostrar** que el mecanismo de monitoreo detecta problemas. Los archivos de salida llevan el sufijo `_SIMULADO` y la columna `tipo_dato = SIMULADO`.

## 6.1 Indicadores, umbrales y responsables

| Indicador | Cómo se mide | Umbral de alerta | Frecuencia | Responsable | Acción |
|---|---|---|---|---|---|
| **Desempeño**: razón del error relativo | MAE en escala log de la ventana / MAE log de validación (el MAE en hectáreas es muy volátil por los pocos municipios extremos) | > 1,15 (15 % peor) | Mensual (cuando haya datos reales del año cerrado: anual) | Analista de datos | Investigar; evaluar reentrenar |
| **Deriva de datos**: PSI por variable | *Population Stability Index* vs. datos de entrenamiento | 0,10-0,25 vigilar · ≥ 0,25 alerta | Mensual | Analista de datos | Revisar origen del cambio |
| **Deriva estadística**: prueba KS | Kolmogorov-Smirnov de cada variable vs. entrenamiento | p < 0,01 | Mensual | Analista de datos | Confirmar con PSI |
| **Calidad de entrada** | % de solicitudes rechazadas por validación | > 10 % | Semanal | Administrador | Revisar formulario/instrucciones |
| **Cobertura del intervalo** | % de valores reales dentro del rango | fuera de 40-60 % | Anual | Analista de datos | Recalibrar σ |
| **Uso y latencia** | Solicitudes por día y tiempo de respuesta | latencia > 2 s | Semanal | Administrador | Revisar despliegue |
| **Satisfacción** | Calificación media en el formulario de feedback | < 3,5 / 5 | Trimestral | Equipo del proyecto | Mejoras de interfaz/contenido |

## 6.2 Cómo se detectan la deriva y la degradación
* **Deriva de datos (*data drift*)**: compara la distribución de cada variable de entrada reciente con la de entrenamiento mediante **PSI** (cuantiles de entrenamiento como cortes) y la prueba **KS**.
* **Degradación del modelo**: cuando se conoce la pérdida real del año cerrado, se calcula el error relativo (MAE en escala log) y se compara con el de referencia (razón del error relativo).
* **Deriva de concepto**: si el MAE sube aunque el PSI sea bajo, cambió la *relación* entre variables y objetivo (p. ej., nuevas políticas o dinámicas de deforestación).

In [ ]:
def psi(ref, nuevo, bins=5):
    # Population Stability Index con cortes por cuantiles de la referencia (entrenamiento)
    ref, nuevo = np.asarray(ref, float), np.asarray(nuevo, float)
    cortes = np.unique(np.percentile(ref, np.linspace(0, 100, bins + 1)))
    if len(cortes) < 3:
        cortes = np.array([-np.inf, np.median(ref), np.inf])
    cortes[0], cortes[-1] = -np.inf, np.inf
    a = np.histogram(ref, cortes)[0] / len(ref)
    b = np.histogram(nuevo, cortes)[0] / len(nuevo)
    a, b = np.clip(a, 1e-4, None), np.clip(b, 1e-4, None)
    return float(np.sum((b - a) * np.log(b / a)))

def deriva_ventana(ventana):
    filas = {}
    for v in FEATURES:
        filas[v] = {"psi": psi(X_dev[v], ventana[v]), "ks_p": float(stats.ks_2samp(X_dev[v], ventana[v]).pvalue)}
    return pd.DataFrame(filas).T

MAE_REF = float(mejor_fila["MAE CV (ha)"])
MAELOG_REF = float(np.mean(np.abs(res_oof)))   # error relativo de referencia (escala log) a partir de la validación cruzada
UMBRAL_MAE, PSI_VIGILAR, PSI_ALERTA = 1.15, 0.10, 0.25

rng = np.random.default_rng(SEED)
N_VENTANA, N_LOTES = 120, 8
registro, mapa_psi = [], {}
for lote in range(1, N_LOTES + 1):
    nivel = 0 if lote <= 4 else lote - 4                       # lotes 1-4 normales · 5-8 con deriva creciente (SIMULADA)
    v = test_df.iloc[rng.choice(len(test_df), N_VENTANA, replace=True)].copy()
    real = v["target"].to_numpy(float)
    v["cattle_heads"] = v["cattle_heads"] * (1 + 0.25 * nivel)
    v["coca_ha"] = v["coca_ha"] * (1 + 0.5 * nivel)
    v["nbi_pct"] = np.clip(v["nbi_pct"] + 3 * nivel, 0, 100)
    real_sim = real * (1 + 0.20 * nivel)                       # la pérdida real simulada se acelera → el modelo queda desactualizado
    pred_log = mejor_pipe.predict(v[FEATURES])
    pred = np.clip(np.expm1(pred_log), 0, None)
    mae = float(mean_absolute_error(real_sim, pred))
    mae_log = float(np.mean(np.abs(np.log1p(real_sim) - pred_log)))
    dv = deriva_ventana(v)
    mapa_psi[f"Lote {lote}"] = dv["psi"]
    razon, psi_max = mae_log / MAELOG_REF, float(dv["psi"].max())
    estado = "ALERTA" if (razon > UMBRAL_MAE or psi_max >= PSI_ALERTA) else ("VIGILAR" if psi_max >= PSI_VIGILAR else "OK")
    registro.append({"lote": lote, "fecha_simulada": f"2027-{lote:02d}-01", "n": N_VENTANA, "mae_ha": round(mae, 1), "mae_log": round(mae_log, 3),
                     "razon_error_rel": round(razon, 2), "psi_max": round(psi_max, 3), "variable_psi_max": dv["psi"].idxmax(),
                     "n_vars_ks_p<0.01": int((dv["ks_p"] < 0.01).sum()), "estado": estado, "tipo_dato": "SIMULADO"})
reg = pd.DataFrame(registro)
reg.to_csv("logs/monitoreo_lotes_SIMULADO.csv", index=False)
display(reg)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
colores = {"OK": VERDE, "VIGILAR": DORADO, "ALERTA": ROJO}
ax[0].bar(reg["lote"], reg["psi_max"], color=[colores[e] for e in reg["estado"]])
ax[0].axhline(PSI_VIGILAR, ls=":", color=DORADO); ax[0].axhline(PSI_ALERTA, ls="--", color=ROJO)
ax[0].set_xlabel("lote"); ax[0].set_ylabel("PSI máximo entre variables"); ax[0].set_title("Deriva de datos (SIMULADO)")
ax[1].plot(reg["lote"], reg["razon_error_rel"], marker="o", color=VERDE_OSC); ax[1].axhline(UMBRAL_MAE, ls="--", color=ROJO)
ax[1].set_xlabel("lote"); ax[1].set_ylabel("error relativo ventana / referencia"); ax[1].set_title("Degradación del desempeño (SIMULADO)")
M = pd.DataFrame(mapa_psi).loc[FEATURES]
im = ax[2].imshow(M.values, cmap="YlOrRd", vmin=0, vmax=0.5, aspect="auto")
ax[2].set_yticks(range(len(FEATURES))); ax[2].set_yticklabels(FEATURES, fontsize=8)
ax[2].set_xticks(range(N_LOTES)); ax[2].set_xticklabels(range(1, N_LOTES + 1)); ax[2].grid(False)
ax[2].set_title("PSI por variable y lote (SIMULADO)"); plt.colorbar(im, ax=ax[2], fraction=0.046)
savefig("fig16_monitoreo_simulado")

RESULTS["monitoreo"] = {"datos": "SIMULADOS", "lotes": N_LOTES, "ventana": N_VENTANA, "mae_referencia_ha": round(MAE_REF, 1), "mae_log_referencia": round(MAELOG_REF, 3),
                        "umbrales": {"razon_mae": UMBRAL_MAE, "psi_vigilar": PSI_VIGILAR, "psi_alerta": PSI_ALERTA},
                        "estados": reg["estado"].value_counts().to_dict(),
                        "primer_lote_alerta": int(reg[reg.estado == "ALERTA"]["lote"].min()) if (reg.estado == "ALERTA").any() else None}
falsas = reg[(reg.lote <= 4) & (reg.estado == "ALERTA")]
deteccion = reg[(reg.lote >= 5) & (reg.estado == "ALERTA")]
primera = int(deteccion["lote"].min()) if len(deteccion) else None
RESULTS["monitoreo"]["falsas_alertas_lotes_1_4"] = int(len(falsas)); RESULTS["monitoreo"]["primera_alerta_con_deriva"] = primera
mostrar(f"**Resultado de la prueba (datos simulados).** Los lotes 1-4 no tienen deriva artificial y los lotes 5-8 sí (cada vez más fuerte). "
        f"Estados obtenidos: {RESULTS['monitoreo']['estados']}. "
        + (f"La primera **ALERTA** con deriva real se dispara en el lote **{primera}**. " if primera else "**No se detectó la deriva simulada**: los umbrales deben ajustarse. ")
        + (f"Hubo {len(falsas)} falsa(s) alerta(s) en lotes sin deriva: es el costo de umbrales sensibles y justifica exigir dos revisiones consecutivas antes de reentrenar. " if len(falsas) else "No hubo falsas **alertas** en los lotes sin deriva")
        + (f" (sí hubo {int((reg[reg.lote <= 4].estado == 'VIGILAR').sum())} aviso(s) «VIGILAR», que es normal: con ventanas de {N_VENTANA} filas el PSI tiene ruido de muestreo). " if int((reg[reg.lote <= 4].estado == 'VIGILAR').sum()) else ". ")
        + "El mecanismo PSI + error relativo sirve como disparador de revisión, no como decisión automática.")

In [ ]:
# ---- Prueba de registro (logging) de predicciones: SIMULADA ----
RUTA_LOG = "logs/predicciones_SIMULADO.csv"
COLUMNAS = ["timestamp", "usuario_simulado", "estado", "latencia_ms", "estimado_ha", "nivel_alerta", "mensaje", "tipo_dato"] + FEATURES

rng2 = np.random.default_rng(SEED + 1)
muestra = test_df.iloc[rng2.choice(len(test_df), 30, replace=False)]
roles = ["citizen", "authority", "admin"]
with open(RUTA_LOG, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=COLUMNAS); w.writeheader()
    for i, (_, r) in enumerate(muestra.iterrows()):
        datos = dict(loss_last_year=r.lag0, loss_2y_ago=r.lag1, loss_3y_ago=r.lag2, area_km2=r.area_km2, population=r.population,
                     cattle_heads=r.cattle_heads, planted_area_ha=r.planted_area_ha, nbi_pct=r.nbi_pct, coca_ha=r.coca_ha, armed_groups=r.armed_groups)
        if i in (7, 19, 26):                       # tres solicitudes inválidas simuladas (incidentes menores)
            datos["nbi_pct"] = [130, -5, 250][[7, 19, 26].index(i)]
        t0 = time.perf_counter()
        fila = {"timestamp": pd.Timestamp.now().isoformat(timespec="seconds"), "usuario_simulado": roles[i % 3], "tipo_dato": "SIMULADO",
                **{k: datos[n] for k, n in zip(FEATURES, ["loss_last_year", "loss_2y_ago", "loss_3y_ago", "area_km2", "population",
                                                           "cattle_heads", "planted_area_ha", "nbi_pct", "coca_ha", "armed_groups"])}}
        try:
            out = predecir_perdida(cargado, **datos)
            fila.update(estado="ok", estimado_ha=out["estimado_ha"], nivel_alerta=out["nivel_alerta"], mensaje="")
        except ValueError as e:
            fila.update(estado="entrada_invalida", estimado_ha="", nivel_alerta="", mensaje=str(e))
        fila["latencia_ms"] = round((time.perf_counter() - t0) * 1000, 1)
        w.writerow(fila)

log = pd.read_csv(RUTA_LOG)
resumen_log = {"solicitudes": int(len(log)), "invalidas": int((log.estado != "ok").sum()),
               "pct_invalidas": round(float((log.estado != "ok").mean() * 100), 1),
               "latencia_media_ms": round(float(log.latencia_ms.mean()), 1),
               "niveles": log.nivel_alerta.value_counts().to_dict()}
display(log.head(8)); display(log[log.estado != "ok"][["timestamp", "estado", "mensaje"]])
RESULTS["log_prueba"] = resumen_log
mostrar(f"**Prueba de registro (SIMULADA):** {resumen_log['solicitudes']} solicitudes, {resumen_log['invalidas']} rechazadas por validación "
        f"({resumen_log['pct_invalidas']} %), latencia media {resumen_log['latencia_media_ms']} ms. El archivo `{RUTA_LOG}` es la evidencia; "
        "las solicitudes inválidas quedan registradas como incidentes menores con su mensaje de error.")

## 6.3 Frecuencia de revisión, responsables y criterios de reentrenamiento

| Elemento | Definición |
|---|---|
| **Revisión operativa** | Semanal (calidad de entrada, uso, latencia) a cargo del administrador |
| **Revisión del modelo** | Mensual con datos de uso; **anual** al cerrar cada año, cuando se conoce la pérdida real y puede calcularse el MAE verdadero |
| **Responsables** | Analista de datos (modelo y deriva), administrador (plataforma e incidentes), equipo del proyecto (feedback y mejoras) |
| **Criterios de reentrenamiento** | (1) razón del error relativo > 1,15 en dos revisiones consecutivas; **o** (2) PSI ≥ 0,25 en 2 o más variables clave; **o** (3) llegada de un nuevo año de datos oficiales; **o** (4) cobertura del intervalo fuera del 40-60 % |

## 6.4 Riesgos, controles de calidad y procedimiento de actualización

| Riesgo | Control de calidad | Mitigación |
|---|---|---|
| Interpretar el pronóstico como dato oficial | Etiquetado visible "simulado/ilustrativo" | Texto de ayuda y documentación |
| Subestimar municipios con pérdidas extremas | Reporte de sesgo por cuartil (Fase 4) | Comunicar el rango y priorizar revisión humana en alertas altas |
| Deriva de datos | PSI y KS mensuales | Reentrenamiento según criterios |
| Pérdida de reproducibilidad | Semilla fija, versiones de librerías fijadas, notebook en el repositorio | Regenerar el modelo con el notebook |
| Errores de entrada | Validación de rangos y registro de incidentes | Mensajes claros al usuario |

**Procedimiento de actualización del modelo:** (1) incorporar los nuevos datos al CSV; (2) ejecutar este cuaderno completo; (3) comparar el nuevo modelo con el vigente en el mismo conjunto de prueba; (4) si es mejor, reemplazar `models/crispml_pipeline.joblib`, actualizar la versión y el registro de cambios; (5) publicar en Render; (6) si falla, volver al artefacto anterior (control de versiones con Git).

---
# Resumen de consistencia y exportación
Estas cifras salen directamente del código ejecutado y **deben ser idénticas** en el informe PDF, en el repositorio y en la aplicación publicada.

In [ ]:
RESULTS["meta"] = {"proyecto": "Forest Awareness", "semilla": SEED, "sklearn": sklearn.__version__, "pandas": pd.__version__,
                   "dataset": "SIMULADO", "repositorio": "https://github.com/scarletnatalianio23-source/Proyect_machine_learning-",
                   "aplicacion": "https://forest-awareness-4qpy.onrender.com/", "generado": pd.Timestamp.now().isoformat(timespec="seconds")}
RESULTS["cv_tabla"] = EXP.to_dict("records")
RESULTS["referentes_cv"] = {k: round(v, 1) for k, v in BASELINES_CV.items()}

def _json_seguro(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)

with open("models/crispml_results.json", "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, indent=2, ensure_ascii=False, default=_json_seguro)

resumen = pd.DataFrame([
    ["Dataset", f"{RESULTS['dataset']['filas']:,} filas · {RESULTS['dataset']['municipios']} municipios · {RESULTS['dataset']['anios'][0]}-{RESULTS['dataset']['anios'][1]} (SIMULADO)"],
    ["Tarea", "Regresión: pérdida de bosque (ha) del año siguiente"],
    ["Partición", f"Dev (2018-2022): {RESULTS['particion']['dev_filas']} filas · CV temporal 2020/2021/2022 · Prueba 2023: {RESULTS['particion']['test_filas']} filas"],
    ["PCA", f"{RESULTS['pca']['n_variables']} variables → {RESULTS['pca']['n95']} componentes (≥95 % varianza); PC1 = {RESULTS['pca']['varianza_pc1']} %"],
    ["Experimentos", f"{RESULTS['experimentos']['n_experimentos']} configuraciones · {RESULTS['experimentos']['ajustes_totales']} ajustes"],
    ["Modelo seleccionado", f"{RESULTS['modelo_seleccionado']['modelo']} · {RESULTS['modelo_seleccionado']['reduccion']} · {RESULTS['modelo_seleccionado']['parametros']}"],
    ["MAE en validación cruzada", f"{RESULTS['modelo_seleccionado']['mae_cv']:.1f} ha"],
    ["MAE / RMSE en PRUEBA 2023", f"{RESULTS['prueba']['mae']:.1f} ha / {RESULTS['prueba']['rmse']:.1f} ha"],
    ["R² (log) / R² (ha) en PRUEBA", f"{RESULTS['prueba']['r2_log']:.3f} / {RESULTS['prueba']['r2_ha']:.3f}"],
    ["Cobertura del intervalo (prueba)", f"{RESULTS['prueba']['cobertura_intervalo']} %"],
    ["Criterios de éxito", str(RESULTS["prueba"]["criterios"])],
    ["Archivo del modelo", f"{RESULTS['despliegue']['archivo']} ({RESULTS['despliegue']['tamano_mb']} MB, scikit-learn {sklearn.__version__})"],
], columns=["Elemento", "Valor"])
display(resumen)
resumen.to_csv("reports/tables/resumen_consistencia.csv", index=False)

# ---- Exportar todo en un .zip ----
SALIDAS = ["static/figures", "reports/tables", "logs"]
ARCHIVOS_MODELO = ["models/crispml_pipeline.joblib", "models/crispml_results.json"]
with zipfile.ZipFile("crispml_resultados.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for carpeta in SALIDAS:
        for raiz, _, archivos in os.walk(carpeta):
            for a in archivos:
                z.write(os.path.join(raiz, a))
    for a in ARCHIVOS_MODELO:
        z.write(a)
print("Creado: crispml_resultados.zip", f"({os.path.getsize('crispml_resultados.zip')/1e6:.1f} MB)")
try:
    from google.colab import files
    files.download("crispml_resultados.zip")
except Exception:
    print("(Fuera de Colab: el .zip queda en el directorio de trabajo.)")

## Conclusiones y limitaciones

* Se aplicó **CRISP-ML(Q)** de punta a punta: comprensión del problema, preparación sin fuga de datos, comparación de tres algoritmos con y sin **PCA**, validación **temporal**, evaluación final en un conjunto de **prueba independiente (2023)**, despliegue con *pipeline* guardado y plan de monitoreo con una prueba simulada.
* **PCA y t-SNE** se usaron para entender la estructura (varianza, cargas, vecindades) y para medir el efecto de la reducción de dimensionalidad en el modelo (sección 4.4).
* **Limitación principal:** el dataset es **simulado**; las métricas describen el método, no la deforestación real. Antes de cualquier uso real hay que reentrenar con datos oficiales (IDEAM/SMByC, DANE) y repetir la validación.
* **Honestidad sobre el desempeño:** los resultados se comparan siempre contra referentes ingenuos; si la mejora sobre «igual al año pasado» es pequeña, se informa tal cual (secciones 4.1 y 4.3).
* **Trabajo pendiente de las personas:** completar la revisión con una persona experta (sección 4.5) y verificar que cifras e imágenes coincidan en el informe, el repositorio y la aplicación.